In [8]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import folium
import time

# Charger les données depuis le fichier Excel
fichier_excel = 'C:/Users/hugoj/Desktop/garages_vendezvotrevoiture_branches_wrapper.xlsx'
df = pd.read_excel(fichier_excel)

# Initialiser le géolocalisateur avec un timeout plus élevé
geolocator = Nominatim(user_agent="garage_locator", timeout=10)

# Créer une carte centrée sur la France
map_france = folium.Map(location=[46.603354, 1.888334], zoom_start=6)

# Fonction pour obtenir la localisation avec gestion des exceptions
def geocode_address(adresse):
    try:
        return geolocator.geocode(adresse)
    except GeocoderTimedOut:
        print(f"Timeout pour l'adresse : {adresse}. Nouvelle tentative...")
        return geocode_address(adresse)  # Réessayer en cas de timeout

# Itérer sur chaque ligne du DataFrame pour obtenir les coordonnées géographiques et ajouter les points sur la carte
for index, row in df.iterrows():
    nom_garage = row['Nom du Garage']
    adresse = row['Adresse']
    
    # Géolocaliser l'adresse pour obtenir la latitude et la longitude
    location = geocode_address(adresse)
    
    if location:
        # Ajouter un marqueur pour chaque garage avec un popup affichant le nom et l'adresse
        folium.Marker(
            location=[location.latitude, location.longitude],
            popup=f"<b>{nom_garage}</b><br>{adresse}",
            tooltip=nom_garage
        ).add_to(map_france)
        
        # Ajouter un cercle vert transparent de 20 km autour de chaque garage
        folium.Circle(
            location=[location.latitude, location.longitude],
            radius=10000,  # Rayon en mètres
            color='green',
            fill=True,
            fill_color='green',
            fill_opacity=0.1
        ).add_to(map_france)
    else:
        print(f"Adresse non trouvée pour le garage : {nom_garage} ({adresse})")

    # Pause de 1 seconde entre les requêtes pour éviter les limitations
    time.sleep(1)

# Sauvegarder la carte dans un fichier HTML
map_france.save("C:/Users/hugoj/Desktop/carte_garages.html")
print("La carte a été créée et enregistrée sous 'C:/Users/hugoj/Desktop/carte_garages.html'. Ouvrez ce fichier dans un navigateur pour visualiser la carte.")


La carte a été créée et enregistrée sous 'C:/Users/hugoj/Desktop/carte_garages.html'. Ouvrez ce fichier dans un navigateur pour visualiser la carte.
